# FTS Parameter Sweep & Out-of-Sample Backtest Evaluation

This notebook evaluates hyperparameter and architectural variations under **controlled isolation** (*ceteris paribus*) for any selected parameter sweep specification (`SweepSpec`).

### Core Principles:
1. **Fixed Baseline Hyperparameters:** All non-swept hyperparameters (learning rate, batch size, dropout, execution fees, slippage) are held strictly constant.
2. **Out-of-Sample Evaluation:** Models are trained on the training split, registered as candidate ONNX models in `ModelRegistryLog`, and backtested on an independent holdout test split using the `BacktestEngine`.
3. **Parameter Sensitivity Curve:** We pair validation statistical fit (`Val IC`) with Out-of-Sample trading metrics (`OOS Sharpe`, `Max Drawdown`) across the swept parameter (`spec.sweep_param`) to pinpoint model capacity sweet-spots and detect overfitting.

### 1. Import Dependencies & Set Pathing

In [ ]:
import os
import sys
import logging
import pandas as pd
import matplotlib.pyplot as plt

# Ensure src and project modules are on path
sys.path.insert(0, os.path.abspath("../src"))

from trading_bot.config import settings
from plugins.nets.spec import SweepSpec
from plugins.nets.training.sweep_runner import run_parameter_sweep

# Set logging level
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

### 2. Load Sweep Specification & Execute Controlled Sweep

Set `spec_path` to point to any parameter sweep specification YAML file (e.g., `specs/train/BTCUSDT/sweep/lstm_num_layers.yaml` or `specs/train/BTCUSDT/sweep/lstm_hidden_dim.yaml`). The sweep runner dynamically inspects `spec.sweep_param` and `spec.sweep_values`.

In [ ]:
spec_path = "../specs/train/BTCUSDT/sweep/lstm_num_layers.yaml"
spec = SweepSpec.from_yaml(spec_path)

print(f"Loaded Sweep Spec     : '{spec.sweep_name}'")
print(f"Model Architecture    : {spec.model_type.upper()}")
print(f"Target Market         : {spec.market.market_id} ({spec.market.interval})")
print(f"Sweeping Parameter    : '{spec.sweep_param}' over {spec.sweep_values}")
print(f"Train Date Range      : {spec.train_dates.start_date} to {spec.train_dates.end_date}")
print(f"Test Date Range       : {spec.test_dates.start_date} to {spec.test_dates.end_date}")

results_df = run_parameter_sweep(spec)

print(f"\n=================== OUT-OF-SAMPLE SWEEP RESULTS ({spec.sweep_param}) ===================")
display(results_df)

### 3. Sensitivity & Out-of-Sample Performance Analysis

We plot **Validation IC (Statistical Fit)** alongside **Out-of-Sample Sharpe Ratio (Strategy Performance)** across the swept parameter (`spec.sweep_param`) to evaluate parameter sensitivity and capacity thresholds.

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))

sweep_param = spec.sweep_param
param_label = sweep_param.replace('_', ' ').title()
model_name = spec.model_type.upper()
x_vals = results_df[sweep_param]

color = 'tab:blue'
ax1.set_xlabel(f'{model_name} {param_label}', fontsize=12)
ax1.set_ylabel('Validation IC', color=color, fontsize=12)
ax1.plot(x_vals, results_df['val_ic'], color=color, marker='o', linewidth=2, label='Val IC')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('OOS Sharpe Ratio', color=color, fontsize=12)
ax2.plot(x_vals, results_df['oos_sharpe'], color=color, marker='s', linewidth=2, linestyle='--', label='OOS Sharpe')
ax2.tick_params(axis='y', labelcolor=color)

plt.title(f'{model_name} {param_label} Sensitivity vs. Strategy Performance', fontsize=14, pad=15)
fig.tight_layout()
plt.show()